In [4]:
import pandas as pd
import geopandas as gpd
import re
from pathlib import Path

DATA = Path("../../data")   # same as 2018, adjust if your notebook sits elsewhere

# Boundary + population — SAME files you already have, no re-download
BEF_PATH = DATA / "oh_cong_adopted_2025/October 31 2025 CD BAF.xlsx"
P1_PATH  = DATA / "oh_pl2020_b/oh_pl2020_p1_b.shp"

# NEW: 2022 precinct results + geometry (no_splits version)
VOTES_2022_PATH = DATA / "oh_gen_2022_prec/oh_2022_gen_prec_no_splits.shp"  # adjust folder/filename to match your download

TARGET_DISTRICT = 15
pd.set_option("display.max_columns", None)

In [5]:
votes22 = gpd.read_file(VOTES_2022_PATH)
print("Shape:", votes22.shape)
print("CRS:", votes22.crs)

# Find the U.S. House columns (GCON##D / GCON##R pattern)
gcon_cols = [c for c in votes22.columns if c.startswith("GCON")]
print(f"\n{len(gcon_cols)} GCON (US House) columns found:")
print(gcon_cols)

# Show the non-vote identifier columns too
id_cols = [c for c in votes22.columns if not c.startswith(("G22","GCON","GSL","GSU"))]
print(f"\nID/geometry columns: {id_cols}")

Shape: (8941, 256)
CRS: epsg:4269

30 GCON (US House) columns found:
['GCON01DLAN', 'GCON01RCHA', 'GCON02DMEA', 'GCON02RWEN', 'GCON03DBEA', 'GCON03RSTA', 'GCON04DWIL', 'GCON04RJOR', 'GCON05DSWA', 'GCON05RLAT', 'GCON06DLYR', 'GCON06RJOH', 'GCON07DDIE', 'GCON07RMIL', 'GCON08DENO', 'GCON08RDAV', 'GCON09DKAP', 'GCON09RMAJ', 'GCON10DESR', 'GCON10RTUR', 'GCON11DBRO', 'GCON11RBRE', 'GCON12DRIP', 'GCON12RBAL', 'GCON13DSYK', 'GCON13RGIL', 'GCON14DKIL', 'GCON14RJOY', 'GCON15DJOS', 'GCON15RCAR']

ID/geometry columns: ['UNIQUE_ID', 'COUNTYFP', 'COUNTYNM', 'PRECINCT', 'PRECCODE', 'VTDST22', 'geometry']


In [7]:
# Sum all D House columns and all R House columns per precinct.
# Each precinct was in one old district, so only that district's column is nonzero —
# summing across all districts gives total House D / R per precinct regardless of old district.

d_cols = [c for c in gcon_cols if c[6] == "D"]   # GCON + 2 district digits, then party at index 6
r_cols = [c for c in gcon_cols if c[6] == "R"]
print(f"D columns ({len(d_cols)}):", d_cols)
print(f"R columns ({len(r_cols)}):", r_cols)

# Make sure vote columns are numeric
votes22[gcon_cols] = votes22[gcon_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

votes22["H22_DEM"] = votes22[d_cols].sum(axis=1)
votes22["H22_REP"] = votes22[r_cols].sum(axis=1)
votes22["H22_TOT"] = votes22["H22_DEM"] + votes22["H22_REP"]

# Validation: statewide totals should match certified 2022 Ohio US House results
print("\nStatewide 2022 US House totals (summed across all precincts):")
print(votes22[["H22_DEM","H22_REP","H22_TOT"]].sum())

D columns (15): ['GCON01DLAN', 'GCON02DMEA', 'GCON03DBEA', 'GCON04DWIL', 'GCON05DSWA', 'GCON06DLYR', 'GCON07DDIE', 'GCON08DENO', 'GCON09DKAP', 'GCON10DESR', 'GCON11DBRO', 'GCON12DRIP', 'GCON13DSYK', 'GCON14DKIL', 'GCON15DJOS']
R columns (15): ['GCON01RCHA', 'GCON02RWEN', 'GCON03RSTA', 'GCON04RJOR', 'GCON05RLAT', 'GCON06RJOH', 'GCON07RMIL', 'GCON08RDAV', 'GCON09RMAJ', 'GCON10RTUR', 'GCON11RBRE', 'GCON12RBAL', 'GCON13RGIL', 'GCON14RJOY', 'GCON15RCAR']

Statewide 2022 US House totals (summed across all precincts):
H22_DEM    1790614.0
H22_REP    2318993.0
H22_TOT    4109607.0
dtype: float64


In [8]:
blocks = gpd.read_file(P1_PATH)
print("Blocks shape:", blocks.shape)
print("CRS:", blocks.crs)

# Also load the BEF and build the OH-15 block list (same as 2018)
bef = pd.read_excel(BEF_PATH, dtype=str)
oh15_blocks = bef.loc[bef["DistrictID:1"].astype(str).str.strip() == str(TARGET_DISTRICT), "Block"].astype(str).str.strip()
print(f"OH-15 blocks: {len(oh15_blocks):,}")

Blocks shape: (276428, 77)
CRS: epsg:4269
OH-15 blocks: 16,658


In [9]:
# Step 1: assign each block to the 2022 precinct that contains its centroid.
oh15_fips3 = ['023','047','049','071','097','109','129']
blocks_7 = blocks[blocks["GEOID20"].str[2:5].isin(oh15_fips3)].copy()
print(f"Blocks in the 7 OH-15 counties: {len(blocks_7):,}")

# Centroid in projected CRS, then back to 4269 (same as 2018)
blocks_7 = blocks_7.to_crs(3857)
blocks_7["centroid"] = blocks_7.geometry.centroid
blocks_7 = blocks_7.set_geometry("centroid").to_crs(4269)

# Spatial join: which 2022 precinct contains each block centroid?
block_prec = gpd.sjoin(
    blocks_7[["GEOID20","P0010001","centroid"]],
    votes22[["UNIQUE_ID","COUNTYFP","PRECINCT","H22_DEM","H22_REP","H22_TOT","geometry"]],
    how="left", predicate="within")

print(f"\nBlocks joined: {len(block_prec):,}")
matched = block_prec["UNIQUE_ID"].notna().sum()
print(f"Blocks that landed in a precinct: {matched:,}")
print(f"Blocks with NO precinct: {len(block_prec)-matched:,}")
lost_pop = block_prec[block_prec["UNIQUE_ID"].isna()]["P0010001"].sum()
print(f"Population in unmatched blocks: {lost_pop:,}")

Blocks in the 7 OH-15 counties: 31,191

Blocks joined: 31,193
Blocks that landed in a precinct: 31,179
Blocks with NO precinct: 14
Population in unmatched blocks: 8


In [10]:
# Guard: the sjoin created 2 duplicate block rows (block centroid inside overlapping precincts).
# Keep one row per block (first match) so votes aren't double-counted.
before = len(block_prec)
block_prec = block_prec[~block_prec.index.duplicated(keep="first")].copy()
print(f"Dropped {before - len(block_prec)} duplicate block row(s); now {len(block_prec):,}")

# Step 2: population-weighted vote split (same as 2018)
prec_pop = block_prec.groupby("UNIQUE_ID")["P0010001"].transform("sum")
block_prec["pop_share"] = block_prec["P0010001"] / prec_pop.replace(0, pd.NA)

# 0-population precincts: equal split across blocks so votes aren't lost
n_blocks = block_prec.groupby("UNIQUE_ID")["GEOID20"].transform("size")
block_prec["pop_share"] = block_prec["pop_share"].fillna(1.0 / n_blocks)

for col in ["H22_DEM","H22_REP","H22_TOT"]:
    block_prec[col.replace("H22_","blk_")] = block_prec[col] * block_prec["pop_share"]

# Validation: block-level votes should sum back to precinct totals (7 counties)
# Compare against votes22 restricted to the 7 counties
v22_7 = votes22[votes22["COUNTYFP"].astype(str).str.zfill(3).isin(oh15_fips3)]
print("\n7-county precinct totals (votes22):")
print(v22_7[["H22_DEM","H22_REP","H22_TOT"]].sum())
print("\n7-county block totals (after split):")
print(block_prec[["blk_DEM","blk_REP","blk_TOT"]].sum())

Dropped 2 duplicate block row(s); now 31,191

7-county precinct totals (votes22):
H22_DEM    306373.0
H22_REP    256824.0
H22_TOT    563197.0
dtype: float64

7-county block totals (after split):
blk_DEM    306592.0
blk_REP    257763.0
blk_TOT    564355.0
dtype: float64


In [11]:
# Proper validation: do the block votes sum back to EACH precinct's original total?
# This is the real conservation test (independent of county-boundary filtering).
check = block_prec.groupby("UNIQUE_ID")[["blk_DEM","blk_REP","blk_TOT"]].sum()
orig = votes22.set_index("UNIQUE_ID")[["H22_DEM","H22_REP","H22_TOT"]]

merged = check.join(orig, how="left")
merged["dem_diff"] = (merged["blk_DEM"] - merged["H22_DEM"]).abs()
merged["rep_diff"] = (merged["blk_REP"] - merged["H22_REP"]).abs()

print(f"Precincts checked: {len(merged):,}")
print(f"Max DEM diff for any precinct: {merged['dem_diff'].max():.4f}")
print(f"Max REP diff for any precinct: {merged['rep_diff'].max():.4f}")
print(f"Total abs DEM diff across all precincts: {merged['dem_diff'].sum():.2f}")
print(f"\nPrecincts with diff > 1 vote (should be none if split is clean):")
print(merged[merged["dem_diff"] > 1][["blk_DEM","H22_DEM","blk_REP","H22_REP"]].head(10))

Precincts checked: 1,173
Max DEM diff for any precinct: 0.0000
Max REP diff for any precinct: 0.0000
Total abs DEM diff across all precincts: 0.00

Precincts with diff > 1 vote (should be none if split is clean):
Empty DataFrame
Columns: [blk_DEM, H22_DEM, blk_REP, H22_REP]
Index: []


In [12]:
# Step 3: keep only blocks in the new OH-15, then sum.
oh15_set = set(oh15_blocks)
in_oh15 = block_prec[block_prec["GEOID20"].isin(oh15_set)].copy()
print(f"Blocks in new OH-15: {len(in_oh15):,} (of {len(block_prec):,} in the 7 counties)")

oh15_2022_result = in_oh15[["blk_DEM","blk_REP","blk_TOT"]].sum()
print("\n=== 2022 U.S. House vote, re-aggregated onto NEW OH-15 ===")
print(oh15_2022_result.round(0))

d = oh15_2022_result["blk_DEM"]
r = oh15_2022_result["blk_REP"]
two_party_d = d / (d + r) * 100
print(f"\n2022 two-party D share in new OH-15: {two_party_d:.1f}%  (R: {100-two_party_d:.1f}%)")

Blocks in new OH-15: 16,658 (of 31,191 in the 7 counties)

=== 2022 U.S. House vote, re-aggregated onto NEW OH-15 ===
blk_DEM    106092.0
blk_REP    141081.0
blk_TOT    247173.0
dtype: float64

2022 two-party D share in new OH-15: 42.9%  (R: 57.1%)


In [17]:
# District composition read straight from the data. The BEF (official adopted map) defines
# which blocks are in OH-15; this just names the counties those blocks fall in.
in_oh15["cty3"] = in_oh15["GEOID20"].str[2:5]
name_map = (votes22.assign(cty3=votes22["COUNTYFP"].astype(str).str.zfill(3))
            .groupby("cty3")["COUNTYNM"].first())
print("New OH-15 spans these counties (from the BEF block assignment):")
for f in sorted(in_oh15["cty3"].unique()):
    print(f"  {name_map.get(f, '??')} ({f})")

New OH-15 spans these counties (from the BEF block assignment):
  CLARK (023)
  FAYETTE (047)
  FRANKLIN (049)
  HIGHLAND (071)
  MADISON (097)
  MIAMI (109)
  PICKAWAY (129)


In [18]:
# Save 2022 OH-15 block-level result for the baseline table
export_2022 = in_oh15[["GEOID20","cty3","UNIQUE_ID","PRECINCT","blk_DEM","blk_REP","blk_TOT"]].copy()
export_2022.columns = ["GEOID20","cty3","unique_id_2022","precinct_2022",
                       "dem_2022","rep_2022","tot_2022"]
export_2022.to_csv("../../data/derived_oh15_2022_blocks.csv", index=False)
print(f"Saved {len(export_2022):,} blocks. 2022 OH-15 totals:")
print(export_2022[["dem_2022","rep_2022","tot_2022"]].sum())

Saved 16,658 blocks. 2022 OH-15 totals:
dem_2022    106092.106101
rep_2022    141081.238792
tot_2022    247173.344893
dtype: float64
